In [1]:
import pandas as pd 
import numpy as np 

In [2]:
train_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/test.csv')

# Housing Price Machine Learning - Linear Regression

In [22]:
#Import the requisite packages from sklearn 
#For splitting and cross validation
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV

#for the pipeling 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

#evaluation metrics 
from sklearn.metrics import mean_squared_error, mean_absolute_error 

#to actually run the regressions 
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [9]:
#transform the dependent variable and get my train and test X 
y = np.log1p(train_df['SalePrice'])
X = train_df.drop(columns = ['SalePrice'])
X_test = test_df.copy()

In [25]:
#preprocess the data 
numeric_features = ['LotFrontage','LotArea','MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','1stFlrSF',
                   '2ndFlrSF','LowQualFinSF','GrLivArea','BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr',
                   'KitchenAbvGr','TotRmsAbvGrd','Fireplaces','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch',
                   '3SsnPorch','ScreenPorch','PoolArea','MiscVal']

categorical_features = ['MSSubClass','MSZoning','Street','Alley','LotShape','LandContour','Utilities','LotConfig',
                       'LandSlope', 'Neighborhood', 'Condition1','Condition2','BldgType','HouseStyle','OverallQual',
                       'OverallCond','YearBuilt','YearRemodAdd','RoofStyle','RoofMatl','Exterior1st','Exterior2nd','MasVnrType',
                       'ExterQual','ExterCond','Foundation','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
                       'Heating','HeatingQC','CentralAir','Electrical','KitchenQual','Functional','FireplaceQu','GarageType','GarageYrBlt'
                       , 'GarageFinish', 'GarageQual','GarageCond','PavedDrive','PoolQC','MiscFeature','MoSold','YrSold','SaleType',
                       'SaleCondition']

numeric_transformer = Pipeline(steps = [("imputer",SimpleImputer(strategy = "median")),
                                      ("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "constant", fill_value = "missing")),
                                            ("to_str", FunctionTransformer(lambda x: x.astype(str))),
                                           ("onehot", OneHotEncoder(handle_unknown = "ignore"))])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
    

In [37]:
#Now run the linear regression on a simple 80-20 split. Here i won't care about colinearity but in reality that is important 
#especially for interpretation. 

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.2, random_state = 1234)

linear_regression = Pipeline(steps = [("preprocess", preprocess), 
                            ("model",LinearRegression())])

linear_regression.fit(X_train,y_train)
prediction = linear_regression.predict(X_val)


print("Baseline RMSE OLS (log target):", rmse(y_val, prediction))
print("Baseline MAE OLS (log target):", mean_absolute_error(y_val, prediction))

Baseline RMSE OLS (log target): 0.22735451232253523
Baseline MAE OLS (log target): 0.12622928608560466


In [30]:
# now do it with K-Folds 
cv = KFold(n_splits = 5, shuffle = True, random_state = 1234) 

scoring = {
    "rmse":"neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_results = cross_validate(linear_regression, 
                            X,
                            y,
                            cv=cv,
                            scoring = scoring, 
                            return_train_score = False)

print("5-fold CV RMSE:", (-cv_results["test_rmse"]).mean(), "±", (-cv_results["test_rmse"]).std())
print("5-fold CV MAE: ", (-cv_results["test_mae"]).mean(),  "±", (-cv_results["test_mae"]).std())

5-fold CV RMSE: 0.21036736137805878 ± 0.016084030770292496
5-fold CV MAE:  0.12296551169015042 ± 0.003819771335234957


In [34]:
# Now for submission to Kaggle 
linear_regression.fit(X,y)
test_log_prediction = linear_regression.predict(X_test)
test_prediction = np.expm1(test_log_prediction)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_prediction
})

submission.to_csv("/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/Submission_03022026_LinearRegression.csv", index=False)
submission.head()

,Id,SalePrice
0,1461,119214.966302
1,1462,164285.282541
2,1463,184250.570219
3,1464,206032.501667
4,1465,184230.450204


# Housing Price Machine Learning - L1 Regularization (LASSO)

In [39]:
#Lasso has a parameter we need to tune 
from sklearn.linear_model import LassoCV
lasso_regression_CV = Pipeline(steps = [("preprocess", preprocess), 
                            ("model",LassoCV(
                                alphas = np.logspace(-5,-2,30),
                                cv = 5, 
                                max_iter = 20000,
                                random_state = 1234
                            ))])

lasso_regression_CV.fit(X_train,y_train)
prediction_lasso = lasso_regression_CV.predict(X_val)


print("Baseline RMSE Lasso (log target):", rmse(y_val, prediction_lasso))
print("Baseline MAE  Lasso (log target):", mean_absolute_error(y_val, prediction_lasso))

Baseline RMSE Lasso (log target): 0.1508828537598183
Baseline MAE  Lasso (log target): 0.09771810521056813


In [40]:
#now do KFold 
cv = KFold(n_splits = 5, shuffle = True, random_state = 1234) 

scoring = {
    "rmse":"neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_results_lasso = cross_validate(lasso_regression_CV, 
                            X,
                            y,
                            cv=cv,
                            scoring = scoring, 
                            return_train_score = False)

print("5-fold CV RMSE - Lasso Tuned:", (-cv_results_lasso["test_rmse"]).mean(), "±", (-cv_results_lasso["test_rmse"]).std())
print("5-fold CV MAE - Lasso Tuned: ", (-cv_results_lasso["test_mae"]).mean(),  "±", (-cv_results_lasso["test_mae"]).std())

5-fold CV RMSE - Lasso Tuned: 0.14393395055531272 ± 0.025156789295994825
5-fold CV MAE - Lasso Tuned:  0.0869063283071033 ± 0.006681001812989826


In [41]:
# Now for submission to Kaggle 
lasso_regression_CV.fit(X,y)
test_log_prediction = lasso_regression_CV.predict(X_test)
test_prediction = np.expm1(test_log_prediction)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_prediction
})

submission.to_csv("/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/Submission_03022026_LassoRegression.csv", index=False)
submission.head()

,Id,SalePrice
0,1461,118713.564855
1,1462,152297.161576
2,1463,179174.340298
3,1464,198942.989021
4,1465,194848.767162


# Housing Price Machine Learning - L2 Regularization (RIDGE)

In [49]:
#Ridge has a parameter we need to tune 
from sklearn.linear_model import RidgeCV
Ridge_regression_CV = Pipeline(steps = [("preprocess", preprocess), 
                            ("model",RidgeCV(
                                alphas = np.logspace(-4,4,60),
                                cv = 5
                            ))])

Ridge_regression_CV.fit(X_train,y_train)
prediction_Ridge = Ridge_regression_CV.predict(X_val)
best_ridge_alpha = Ridge_regression_CV.named_steps["model"].alpha_
print(best_ridge_alpha)
print("Baseline RMSE Ridge (log target):", rmse(y_val, prediction_Ridge))
print("Baseline MAE  Ridge (log target):", mean_absolute_error(y_val, prediction_Ridge))

14.208308325339209
0.0001 10000.0
Baseline RMSE Ridge (log target): 0.16012269784289854
Baseline MAE  Ridge (log target): 0.10544748531971836


In [47]:
#now do KFold 
cv = KFold(n_splits = 5, shuffle = True, random_state = 1234) 

scoring = {
    "rmse":"neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_results_ridge = cross_validate(Ridge_regression_CV, 
                            X,
                            y,
                            cv=cv,
                            scoring = scoring, 
                            return_train_score = False)

print("5-fold CV RMSE - Ridge Tuned:", (-cv_results_ridge["test_rmse"]).mean(), "±", (-cv_results_ridge["test_rmse"]).std())
print("5-fold CV MAE - Ridge Tuned: ", (-cv_results_ridge["test_mae"]).mean(),  "±", (-cv_results_ridge["test_mae"]).std())

5-fold CV RMSE - Ridge Tuned: 0.14593351842068253 ± 0.02457769461446352
5-fold CV MAE - Ridge Tuned:  0.09073963556937238 ± 0.007595449486885709


# Housing Price Machine Learning - Regularization (ElasticNet)

In [56]:
#elasticnet has two parameters we need to tune 
from sklearn.linear_model import ElasticNetCV
elasticnet_regression_CV = Pipeline(steps = [("preprocess", preprocess), 
                            ("model",ElasticNetCV(
                                l1_ratio = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0],
                                alphas = np.logspace(-5,0,60),
                                cv = 5,
                                max_iter = 200000,
                                random_state =1234
                            ))])

elasticnet_regression_CV.fit(X_train,y_train)
prediction_elasticnet = elasticnet_regression_CV.predict(X_val)
best_EN_alpha = elasticnet_regression_CV.named_steps["model"].alpha_
best_EN_l1_ratio = elasticnet_regression_CV.named_steps["model"].l1_ratio_


print("Best EN Alpha:", best_EN_alpha)
print("Best EN L1 Ratio:", best_EN_l1_ratio)
print("Baseline RMSE ElasticNet (log target):", rmse(y_val, prediction_elasticnet))
print("Baseline MAE  ElasticNet (log target):", mean_absolute_error(y_val, prediction_elasticnet))

Best EN Alpha: 0.0006020894493336125
Best EN L1 Ratio: 0.8
Baseline RMSE ElasticNet (log target): 0.15764271612228414
Baseline MAE  ElasticNet (log target): 0.09956392167806392


In [55]:
#now do KFold 
cv = KFold(n_splits = 5, shuffle = True, random_state = 1234) 

scoring = {
    "rmse":"neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_results_elasticnet = cross_validate(elasticnet_regression_CV, 
                            X,
                            y,
                            cv=cv,
                            scoring = scoring, 
                            return_train_score = False,
                            n_jobs = -1)

print("5-fold CV RMSE - EN Tuned:", (-cv_results_elasticnet["test_rmse"]).mean(), "±", (-cv_results_elasticnet["test_rmse"]).std())
print("5-fold CV MAE - EN Tuned: ", (-cv_results_elasticnet["test_mae"]).mean(),  "±", (-cv_results_elasticnet["test_mae"]).std())

5-fold CV RMSE - EN Tuned: 0.14406182606255255 ± 0.024980142078412336
5-fold CV MAE - EN Tuned:  0.08756854591274568 ± 0.007900425099952344


In [ ]:
# Now for submission to Kaggle 
elasticnet_regression_CV.fit(X,y)
test_log_prediction = elasticnet_regression_CV.predict(X_test)
test_prediction = np.expm1(test_log_prediction)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_prediction
})

submission.to_csv("/Users/alextoy/Desktop/Kaggle Competitions /Housing Prices/house-prices-advanced-regression-techniques/Submission_03022026_Elasticnet.csv", index=False)
submission.head()